# Walmart Compliance Manual — Hybrid RAG Chatbot (Chroma Cloud edition)

This notebook is the Colab version of `walmart_rag_v2.py`. The pipeline is the same: PyMuPDF extraction → section-based chunking → BM25 + dense retrieval → reciprocal rank fusion → GPT-4o-mini answers with citations. **The one architectural change is that the dense index lives in Chroma Cloud instead of an in-memory local Chroma.**

### Pipeline

```
PDF ──► PyMuPDF spans ──► section chunks ──► walmart_chunks.json
                                │
                ┌───────────────┴───────────────┐
                ▼                               ▼
        BM25 (Colab memory)          Chroma Cloud collection
                │                     (OpenAI embeddings)
                └──────────► RRF ◄──────────────┘
                              │
                              ▼
                 GPT-4o-mini answer + citations
```

### What changes with Chroma Cloud

| | Local (the script) | Chroma Cloud (this notebook) |
|---|---|---|
| Client | `chromadb.Client()`, in memory | `chromadb.CloudClient(...)` |
| Lifetime | lost when the session ends | persists in your Chroma Cloud database |
| Vector index | HNSW | SPANN (Chroma's distributed index) |
| Distance metric | `metadata={"hnsw:space": "cosine"}` | `configuration={"spann": {"space": "cosine"}}` |
| Re-running | collection deleted and re-embedded every run | reused; embeds only when needed |
| Limits | your machine | cloud quotas (e.g. 300 records per write, 16 KB per document) |

### Before you start

1. Create a Chroma Cloud account and a database. From the dashboard, copy your **API key**, **tenant ID** and **database name**.
2. Have an OpenAI API key ready.
3. In Colab, click the 🔑 icon in the left sidebar and add four secrets, each with **Notebook access** switched on:
   `OPENAI_API_KEY`, `CHROMA_API_KEY`, `CHROMA_TENANT`, `CHROMA_DATABASE`

## Step 0 — Install and import

PyMuPDF (`fitz`) reads the PDF, `rank-bm25` does keyword search, `chromadb` talks to Chroma Cloud, and `openai` creates embeddings and answers. `CloudClient` needs a recent `chromadb`, so the install sets a minimum version.

In [ ]:
# Pin exact versions once you have tested this notebook end to end
!pip install -q "chromadb>=1.0.0" pymupdf openai rank-bm25

In [ ]:
import os
import re
import json
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

import requests
import fitz                     # PyMuPDF
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
from rank_bm25 import BM25Okapi

print("chromadb version:", chromadb.__version__)

## Step 0b — Load the keys

The script read keys from a `.env` file. In Colab we read them from **Colab Secrets** and copy them into environment variables, which is where both the OpenAI SDK and Chroma look for them. Nothing is hard-coded, and the cell never prints a key.

Running locally instead of in Colab? Set the same four environment variables and this cell will find them.

In [ ]:
REQUIRED_KEYS = ["OPENAI_API_KEY", "CHROMA_API_KEY", "CHROMA_TENANT", "CHROMA_DATABASE"]

try:
    from google.colab import userdata
    for name in REQUIRED_KEYS:
        if not os.environ.get(name):
            try:
                os.environ[name] = userdata.get(name)
            except Exception:
                pass          # secret missing or notebook access is off
except ImportError:
    pass                      # not in Colab: rely on environment variables

missing = [name for name in REQUIRED_KEYS if not os.environ.get(name)]
for name in REQUIRED_KEYS:
    print(f"  {name:<16} {'❌ missing' if name in missing else '✅ set'}")

if missing:
    raise EnvironmentError(
        f"Missing: {missing}. Add them as Colab Secrets (🔑 in the sidebar) "
        "and switch on 'Notebook access' for each."
    )

## Step 0c — Get the PDF

The cell downloads the manual from Walmart's site. Some sites block requests that don't come from a browser; if the download fails, the cell opens an upload button so you can add the PDF by hand.

In [ ]:
PDF_PATH = "u-s-product-quality-and-compliance-manual.pdf"
PDF_URL = (
    "https://corporate.walmart.com/content/dam/corporate/documents/"
    "suppliers/requirements/compliance-areas/"
    "u-s-product-quality-and-compliance-manual.pdf"
)


def fetch_pdf(path: str, url: str) -> None:
    """Download the manual; if the site blocks the request, fall back to a manual upload."""
    if Path(path).exists():
        print(f"📄 Found {path} ({Path(path).stat().st_size / 1024:.0f} KB)")
        return
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
        r.raise_for_status()
        if not r.content.startswith(b"%PDF"):
            raise ValueError("the response is not a PDF")
        Path(path).write_bytes(r.content)
        print(f"📥 Downloaded {path} ({len(r.content) / 1024:.0f} KB)")
    except Exception as e:
        print(f"⚠️  Download failed ({e}).")
        print(f"   Download it in your browser from:\n   {url}\n   then upload it below.")
        from google.colab import files
        uploaded = files.upload()
        name = next(iter(uploaded))
        Path(name).rename(path)
        print(f"📄 Saved upload as {path}")


fetch_pdf(PDF_PATH, PDF_URL)

## Step 1 — PDF extraction with PyMuPDF

**Why PyMuPDF (`fitz`)?** PyPDF2 is older and struggles with:
- multi-column layouts (it merges columns incorrectly)
- tables (it produces garbled text)
- font-encoded PDFs (it misses characters)

PyMuPDF uses the MuPDF rendering engine, the same engine used by production PDF viewers. It gives you:
- page-by-page text in the correct reading order
- font size and font name for every text span (useful for detecting headers)
- bounding-box coordinates for every span
- clean Unicode output

We use font size to **detect section headers**: Walmart's PDF uses larger, bolder text for the Roman-numeral section titles (I., II., III., …).

In [ ]:
def extract_pdf_with_metadata(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Extract text from PDF page by page using PyMuPDF.
    Returns a list of text blocks with page number and font metadata.

    Each block looks like:
    {
        "page": 1,
        "text": "I. Introduction",
        "font_size": 14.0,
        "font_name": "Arial-Bold",
        "is_bold": True,
        "bbox": (x0, y0, x1, y1)
    }
    """
    doc = fitz.open(pdf_path)
    blocks = []

    print(f"📄 PDF loaded: {doc.page_count} pages")

    for page_num in range(doc.page_count):
        page = doc[page_num]

        # get_text("dict") returns structured block data with font info
        # This is the key PyMuPDF advantage over PyPDF2
        page_dict = page.get_text("dict")

        for block in page_dict["blocks"]:
            # blocks can be "image" type — we only want "text" type
            if block["type"] != 0:
                continue

            for line in block["lines"]:
                for span in line["spans"]:
                    text = span["text"].strip()
                    if not text:
                        continue

                    blocks.append({
                        "page": page_num + 1,
                        "text": text,
                        "font_size": round(span["size"], 1),
                        "font_name": span["font"],
                        "is_bold": "bold" in span["font"].lower() or span["flags"] & 2**4,
                        "bbox": span["bbox"]
                    })
    page_count = doc.page_count
    doc.close()
    print(f"✅ Extracted {len(blocks)} text spans across {page_count} pages")
    return blocks

## Step 2 — Section-based chunking

**Character-based chunking (the V1 approach we replaced)** splits every N characters with overlap. It cuts mid-sentence and mid-paragraph and ignores the document's structure:

```
"...The CAP must be approved before submitting the corrected product for re"
"test. The retest sample must be submitted within 3 months..."
```
The sentence about approval before retest is split across two chunks.

**Section-based chunking (V2)** detects section headers from the PDF's font metadata and splits where the document's authors put section boundaries. A span counts as a header when:
1. it matches a Roman-numeral pattern (`I.`, `II.`, `III.`, …) or a letter-subsection pattern (`A.`, `B.`, …), **and**
2. its font is larger than the document's median (by at least 1 pt) **or** it is bold, **and**
3. it is longer than 5 characters (to skip stray large characters).

Each chunk is then one complete section, a semantically coherent unit. The retriever gets "Failure Management Program" as one clean chunk instead of pieces spread over three character-boundary chunks.

**When sections are too long:** some sections (such as Section III with all the testing cycles) are very long. If a section exceeds `MAX_CHUNK_CHARS`, it is split into overlapping sub-chunks to preserve context at the split boundaries.

In [ ]:
# Regex patterns for Walmart compliance manual header detection
ROMAN_NUMERAL_PATTERN = re.compile(
    r"^(I{1,3}|IV|V|VI{0,3}|IX|X)\.\s+.+", re.IGNORECASE
)
LETTER_SUBSECTION_PATTERN = re.compile(
    r"^[A-Z]\.\s+.{3,}"
)
NUMBERED_SUBSECTION_PATTERN = re.compile(
    r"^\d+\.\s+.{3,}"
)

MAX_CHUNK_CHARS = 1200   # Max characters per chunk before sub-splitting
OVERLAP_CHARS   = 150    # Overlap when sub-splitting long sections


def is_section_header(block: Dict[str, Any], median_font_size: float) -> bool:
    """
    Determine if a text block is a section header.

    Detection logic (all must pass):
    - Font size >= median + 1pt  OR  block is bold
    - Text matches Roman numeral or letter subsection pattern
    - Text is not too short (avoids catching stray large characters)
    """
    text = block["text"]
    font_size = block["font_size"]
    is_bold = block["is_bold"]

    size_is_large = font_size >= median_font_size + 1.0
    matches_pattern = (
        ROMAN_NUMERAL_PATTERN.match(text) or
        LETTER_SUBSECTION_PATTERN.match(text)
    )
    long_enough = len(text) > 5

    return bool(matches_pattern and (size_is_large or is_bold) and long_enough)


def sub_chunk(text: str, section_id: str, header: str,
              max_chars: int, overlap: int) -> List[Dict[str, Any]]:
    """
    Split a long section into overlapping sub-chunks.
    Used when a section exceeds MAX_CHUNK_CHARS.
    """
    sub_chunks = []
    start = 0
    sub_idx = 0

    while start < len(text):
        end = start + max_chars
        content = text[start:end].strip()
        if content:
            sub_chunks.append({
                "id": f"{section_id}-part{sub_idx:02d}",
                "header": header,
                "content": content,
                "sub_chunk": sub_idx,
                "char_start": start,
                "char_end": min(end, len(text)),
            })
            sub_idx += 1
        start += max_chars - overlap

    return sub_chunks

In [ ]:
def chunk_by_sections(
    blocks: List[Dict[str, Any]],
    pdf_name: str
) -> List[Dict[str, Any]]:
    """
    Group PDF text blocks into section-based chunks.

    Algorithm:
    1. Compute median font size across all blocks (baseline for header detection)
    2. Walk through blocks sequentially
    3. When a header is detected → start a new section
    4. Accumulate text spans into the current section
    5. When section text exceeds MAX_CHUNK_CHARS → sub-split with overlap
    6. Return list of chunk dicts with full metadata

    Returns list of chunks:
    {
        "id": "walmart-sec003-part00",
        "content": "B. Brand Types and Supplier Designations...",
        "metadata": {
            "source": "u-s-product-quality-and-compliance-manual.pdf",
            "section_header": "B. Brand Types and Supplier Designations",
            "page_start": 4,
            "page_end": 5,
            "chunk_type": "section"
        }
    }
    """
    # Compute median font size for header detection threshold
    font_sizes = [b["font_size"] for b in blocks if b["text"].strip()]
    font_sizes.sort()
    median_font = font_sizes[len(font_sizes) // 2] if font_sizes else 11.0
    print(f"   Median font size: {median_font}pt (used for header detection)")

    # Group blocks into sections
    sections = []          # list of {header, page_start, page_end, lines[]}
    current_section = {
        "header": "Preamble",
        "page_start": 1,
        "page_end": 1,
        "lines": []
    }

    for block in blocks:
        text = block["text"]
        page = block["page"]

        if is_section_header(block, median_font):
            # Save current section if it has content
            if current_section["lines"]:
                sections.append(current_section)
            # Start new section
            current_section = {
                "header": text,
                "page_start": page,
                "page_end": page,
                "lines": []
            }
        else:
            current_section["lines"].append(text)
            current_section["page_end"] = page

    # Don't forget the last section
    if current_section["lines"]:
        sections.append(current_section)

    print(f"   Detected {len(sections)} sections from headers")

    # Convert sections → chunks (sub-splitting if needed)
    chunks = []
    for sec_idx, section in enumerate(sections):
        full_text = " ".join(section["lines"])
        full_text = re.sub(r"\s+", " ", full_text).strip()

        if not full_text:
            continue

        header = section["header"]
        # Clean section ID for use as ChromaDB document ID
        section_id = f"walmart-sec{sec_idx:03d}"

        base_metadata = {
            "source": pdf_name,
            "section_header": header,
            "page_start": section["page_start"],
            "page_end": section["page_end"],
            "section_index": sec_idx,
            "chunk_type": "section"
        }

        if len(full_text) <= MAX_CHUNK_CHARS:
            # Section fits in one chunk
            chunks.append({
                "id": section_id,
                "content": f"{header}\n\n{full_text}",
                "metadata": {**base_metadata, "sub_chunk": 0, "total_sub_chunks": 1}
            })
        else:
            # Section too long — sub-split with overlap
            sub_chunks = sub_chunk(
                full_text, section_id, header, MAX_CHUNK_CHARS, OVERLAP_CHARS
            )
            for sc in sub_chunks:
                chunks.append({
                    "id": sc["id"],
                    "content": f"{header}\n\n{sc['content']}",
                    "metadata": {
                        **base_metadata,
                        "sub_chunk": sc["sub_chunk"],
                        "total_sub_chunks": len(sub_chunks),
                        "char_start": sc["char_start"],
                        "char_end": sc["char_end"],
                    }
                })

    print(f"✅ Created {len(chunks)} section-based chunks")
    return chunks

## Step 3 — Save chunks locally as JSON

Two reasons to save the chunks:

1. **Transparency.** You can open `walmart_chunks.json` (📁 Files panel on the left) and see exactly what went into Chroma. If retrieval goes wrong, inspect the JSON first: did the chunking split a section correctly? Did header detection work? Is text missing?
2. **Caching.** Extraction and chunking take time. With the JSON saved, later runs skip re-extraction and load it directly — especially important when processing hundreds of SEC filing PDFs.

The JSON is human-readable and maps 1:1 to what Chroma stores.

> **Colab note:** files in `/content` disappear when the runtime restarts, so in Colab this cache only lasts for the current session. The Chroma Cloud collection, by contrast, persists.

In [ ]:
CHUNKS_FILE = "walmart_chunks.json"


def save_chunks(chunks: List[Dict[str, Any]], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)
    size_kb = Path(path).stat().st_size / 1024
    print(f"💾 Saved {len(chunks)} chunks → {path} ({size_kb:.1f} KB)")
    print(f"   Open {path} to inspect chunk content and metadata")


def load_chunks(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        chunks = json.load(f)
    print(f"📂 Loaded {len(chunks)} chunks from {path}")
    return chunks


def preview_chunks(chunks: List[Dict[str, Any]], n: int = 5) -> None:
    """Print a preview of the first n chunks to show what was extracted."""
    print(f"\n{'='*60}")
    print(f"CHUNK PREVIEW (first {n} of {len(chunks)})")
    print(f"{'='*60}")
    for i, chunk in enumerate(chunks[:n]):
        meta = chunk["metadata"]
        print(f"\n[Chunk {i+1}] ID: {chunk['id']}")
        print(f"  Header   : {meta['section_header']}")
        print(f"  Pages    : {meta['page_start']} → {meta['page_end']}")
        print(f"  Sub-chunk: {meta['sub_chunk']} of {meta['total_sub_chunks']}")
        print(f"  Length   : {len(chunk['content'])} chars")
        print(f"  Preview  : {chunk['content'][:120].strip()}...")
    print(f"{'='*60}\n")

### Run Steps 1–3

If `walmart_chunks.json` already exists, it is loaded instead of re-reading the PDF. **Delete the file to force re-extraction** after you change any chunking code.

In [ ]:
if Path(CHUNKS_FILE).exists():
    # Skip re-extraction if chunks were already saved earlier in this session
    print(f"\n📂 Found existing {CHUNKS_FILE} — loading cached chunks")
    print(f"   (Delete {CHUNKS_FILE} to force re-extraction from the PDF)")
    chunks = load_chunks(CHUNKS_FILE)
else:
    print(f"\n📄 Extracting text from PDF: {PDF_PATH}")
    blocks = extract_pdf_with_metadata(PDF_PATH)

    print("\n✂️  Chunking by sections...")
    chunks = chunk_by_sections(blocks, pdf_name=PDF_PATH)

    # Save locally for inspection and caching
    save_chunks(chunks, CHUNKS_FILE)

# Show the first 5 chunks so you can verify extraction quality
preview_chunks(chunks, n=5)

## Step 4 — Data model

Every retriever (BM25, dense, hybrid) returns the same `RetrievedChunk` shape, so results from different methods can be compared and fused. `retrieval_method` records which one produced it.

In [ ]:
@dataclass
class RetrievedChunk:
    chunk_id: str
    content: str
    score: float
    metadata: Dict
    retrieval_method: str   # "sparse_bm25" | "dense_semantic" | "hybrid_rrf"

## Step 5a — Connect to Chroma Cloud

This is the part that changes from the script.

**`CloudClient` instead of `Client()`.** The client connects to one *database* inside your *tenant* (your Chroma Cloud account). Collections in that database persist after the Colab session ends, and other notebooks with the same keys can open them.

**Where the embedding happens.** The OpenAI embedding function still runs *here*, in Colab. Colab sends text to OpenAI, gets vectors back, and uploads the vectors, the text and the metadata to Chroma Cloud. At query time, the question is embedded in Colab too, and only the vector search happens in the cloud.

**A different index.** Local Chroma uses an HNSW index; Chroma Cloud uses SPANN, its distributed index. So the distance metric is set with `configuration={"spann": {"space": "cosine"}}` rather than the local `metadata={"hnsw:space": "cosine"}` form. Chroma Cloud ignores SPANN tuning parameters, and a cell after indexing checks which metric is actually in use.

**Quotas.** A shared cloud service limits each request. The ones that matter here:

| Quota | Value | How this notebook handles it |
|---|---|---|
| Records per write | 300 | writes in batches of 100 |
| Bytes per document | 16,384 | checks every chunk before upload |
| Results per query | 300 | queries ask for a handful |
| Metadata keys per record | 32 | chunks have 10 |

**Don't delete and recreate on every run.** The script deleted the collection at startup, which was harmless in memory. In the cloud it would throw away stored vectors and pay OpenAI to re-create them each time. This notebook reuses the collection and only re-embeds when you set `RESET_COLLECTION = True`.

In [ ]:
COLLECTION_NAME  = "walmart_compliance_v2"
RESET_COLLECTION = False   # True = delete and re-embed (do this once after changing the chunking)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL  = "gpt-4o-mini"

CLOUD_MAX_RECORDS_PER_WRITE = 300
CLOUD_MAX_DOCUMENT_BYTES    = 16_384
WRITE_BATCH_SIZE            = 100

chroma_client = chromadb.CloudClient(
    api_key=os.environ["CHROMA_API_KEY"],
    tenant=os.environ["CHROMA_TENANT"],
    database=os.environ["CHROMA_DATABASE"],
)

existing = [getattr(c, "name", c) for c in chroma_client.list_collections()]
print(f"✅ Connected to Chroma Cloud — database '{os.environ['CHROMA_DATABASE']}'")
print(f"   Collections already in this database: {existing or 'none'}")

## Step 5b — The pipeline: setup and dual indexing

The script defined the whole `WalmartRAGPipeline` class in one block. Here the class is built over several cells: this cell defines setup and indexing, and the next cells attach the retrieval and answering methods one at a time, so each piece can be run and explained on its own. The finished class is the same.

**Dual indexing** puts every chunk into two indexes:
- **BM25** tokenizes each chunk into words and builds an inverted index with IDF weights. It lives in Colab memory and is rebuilt on every run, which is cheap.
- **Chroma Cloud** stores an OpenAI embedding of each chunk. It persists, so if the collection already holds as many records as there are chunks, embedding is skipped. Writes use `upsert` (insert or overwrite by ID), so re-running never creates duplicates.

In [ ]:
class WalmartRAGPipeline:

    def __init__(self, chroma_client, openai_api_key: str, reset: bool = False):
        print("\n🔧 Initializing Walmart RAG Pipeline (Chroma Cloud)...")
        self.openai_client = OpenAI(api_key=openai_api_key)

        # BM25 sparse index state (local, in Colab memory)
        self.bm25_index: Optional[BM25Okapi] = None
        self.chunk_store: List[Dict[str, Any]] = []

        # Chroma Cloud dense index
        self.chroma_client = chroma_client

        # Runs in Colab: text → OpenAI → vectors, which are then uploaded.
        # api_key_env_var names the variable that holds the key, so the
        # collection's saved configuration records the name, not the key.
        self.embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
            api_key_env_var="OPENAI_API_KEY",
            model_name=EMBED_MODEL,
        )

        if reset:
            try:
                self.chroma_client.delete_collection(COLLECTION_NAME)
                print(f"🗑️  Deleted cloud collection '{COLLECTION_NAME}'")
            except Exception:
                pass  # it did not exist yet

        # get_or_create: reuse the persisted collection instead of rebuilding it
        self.collection = self.chroma_client.get_or_create_collection(
            name=COLLECTION_NAME,
            embedding_function=self.embedding_fn,
            configuration={"spann": {"space": "cosine"}},
        )

        print(f"✅ Chroma Cloud collection '{COLLECTION_NAME}' ready "
              f"({self.collection.count()} records already stored)")
        print("✅ BM25 index ready (will build on index_documents)")

    # ── TOKENIZER ──────────────────────────────────────────────────────────────
    def _tokenize(self, text: str) -> List[str]:
        return re.findall(r"\b\w+\b", text.lower())

    # ── DUAL INDEXING ──────────────────────────────────────────────────────────
    def index_documents(self, chunks: List[Dict[str, Any]]) -> None:
        """
        Index chunks into BOTH BM25 (local) and Chroma Cloud.

        BM25:
          - Tokenizes each chunk into words
          - Builds inverted index + IDF weights
          - Lives in Colab memory, rebuilt every run

        Chroma Cloud:
          - Embeds each chunk with OpenAI (in Colab)
          - Uploads vector + text + metadata in batches
          - Persists between sessions, so embedding is skipped when already done
        """
        print(f"\n📥 Indexing {len(chunks)} section-based chunks...")

        self.chunk_store = chunks

        # BM25 — tokenize and build index
        tokenized_corpus = [self._tokenize(chunk["content"]) for chunk in chunks]
        self.bm25_index = BM25Okapi(tokenized_corpus)

        # Chroma Cloud quota: each document may be at most 16,384 bytes
        too_big = [c["id"] for c in chunks
                   if len(c["content"].encode("utf-8")) > CLOUD_MAX_DOCUMENT_BYTES]
        if too_big:
            raise ValueError(
                f"{len(too_big)} chunks exceed Chroma Cloud's {CLOUD_MAX_DOCUMENT_BYTES:,}-byte "
                f"document limit, e.g. {too_big[:5]}. Lower MAX_CHUNK_CHARS."
            )

        stored = self.collection.count()
        if stored == len(chunks):
            print(f"   ☁️  Collection already holds {stored} records — skipping embedding")
        else:
            if stored:
                print(f"   ⚠️  Collection holds {stored} records but there are {len(chunks)} chunks. "
                      "Upserting now; if you changed the chunking, set RESET_COLLECTION = True "
                      "once so stale chunks are removed.")

            # Chroma Cloud quota: at most 300 records per write
            assert WRITE_BATCH_SIZE <= CLOUD_MAX_RECORDS_PER_WRITE
            for i in range(0, len(chunks), WRITE_BATCH_SIZE):
                batch = chunks[i:i + WRITE_BATCH_SIZE]
                self.collection.upsert(
                    documents=[c["content"] for c in batch],
                    ids=[c["id"] for c in batch],
                    metadatas=[c["metadata"] for c in batch]
                )
                print(f"   Embedded + uploaded batch {i // WRITE_BATCH_SIZE + 1} "
                      f"({min(i + WRITE_BATCH_SIZE, len(chunks))}/{len(chunks)} chunks)")

        print(f"\n✅ BM25 vocabulary   : {len(self.bm25_index.idf):,} unique terms")
        print(f"✅ Chroma Cloud vectors: {self.collection.count()} stored")

## Step 5c — Sparse retrieval (BM25)

BM25 scores chunks by **shared words**, weighting rare words more heavily. It is strong on exact terms and acronyms ("SKPPT", "WERCS") and weak on paraphrases. It runs entirely in Colab.

In [ ]:
def sparse_search(self, query: str, k: int = 5) -> List[RetrievedChunk]:
    tokens = self._tokenize(query)
    scores = self.bm25_index.get_scores(tokens)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]

    return [
        RetrievedChunk(
            chunk_id=self.chunk_store[i]["id"],
            content=self.chunk_store[i]["content"],
            score=float(scores[i]),
            metadata=self.chunk_store[i]["metadata"],
            retrieval_method="sparse_bm25"
        )
        for i in top_indices
    ]


WalmartRAGPipeline.sparse_search = sparse_search

## Step 5d — Dense retrieval (Chroma Cloud)

The question is embedded in Colab with the **same** OpenAI model used at index time, and Chroma Cloud returns the nearest stored vectors along with the text and metadata saved next to them. With cosine space, `distance = 1 − similarity`, so lower distance means more similar.

In [ ]:
def dense_search(self, query: str, k: int = 5) -> List[RetrievedChunk]:
    results = self.collection.query(
        query_texts=[query],
        n_results=min(k, self.collection.count())
    )

    retrieved = []
    for chunk_id, content, distance, metadata in zip(
        results["ids"][0],
        results["documents"][0],
        results["distances"][0],
        results["metadatas"][0]
    ):
        similarity = 1.0 - distance   # cosine distance → similarity
        retrieved.append(RetrievedChunk(
            chunk_id=chunk_id,
            content=content,
            score=similarity,
            metadata=metadata,
            retrieval_method="dense_semantic"
        ))

    return retrieved


WalmartRAGPipeline.dense_search = dense_search

## Step 5e — Reciprocal rank fusion and hybrid search

BM25 scores and cosine similarities are on different scales, so they can't simply be added. **Reciprocal rank fusion** ignores the scores and uses only rank positions:

$$\text{RRF}(d) = \sum_{\text{lists}} \frac{1}{k + \text{rank}(d)}, \qquad k = 60$$

A chunk that appears near the top of both lists gets a combined boost.

In [ ]:
def reciprocal_rank_fusion(
    self,
    ranked_lists: List[List[RetrievedChunk]],
    k: int = 60
) -> List[RetrievedChunk]:
    """
    Merge BM25 and Chroma rankings using rank positions only.
    Score = Σ 1 / (k + rank)   for each retriever list.
    Documents appearing in both lists get a combined boost.
    """
    rrf_scores: Dict[str, float] = {}
    registry: Dict[str, RetrievedChunk] = {}

    for ranked_list in ranked_lists:
        for rank_idx, chunk in enumerate(ranked_list, start=1):
            rrf_scores[chunk.chunk_id] = (
                rrf_scores.get(chunk.chunk_id, 0.0) + 1.0 / (k + rank_idx)
            )
            registry[chunk.chunk_id] = chunk

    fused = []
    for chunk_id, rrf_score in sorted(
        rrf_scores.items(), key=lambda x: x[1], reverse=True
    ):
        orig = registry[chunk_id]
        fused.append(RetrievedChunk(
            chunk_id=chunk_id,
            content=orig.content,
            score=rrf_score,
            metadata=orig.metadata,
            retrieval_method="hybrid_rrf"
        ))

    return fused


def hybrid_search(self, query: str, k: int = 5) -> List[RetrievedChunk]:
    sparse  = self.sparse_search(query, k=k)
    dense   = self.dense_search(query, k=k)
    fused   = self.reciprocal_rank_fusion([sparse, dense])

    print(f"   BM25 top  → [{sparse[0].chunk_id}] "
          f"'{sparse[0].metadata.get('section_header', '')[:50]}'")
    print(f"   Dense top → [{dense[0].chunk_id}] "
          f"'{dense[0].metadata.get('section_header', '')[:50]}'")

    return fused[:k]


WalmartRAGPipeline.reciprocal_rank_fusion = reciprocal_rank_fusion
WalmartRAGPipeline.hybrid_search = hybrid_search

## Step 5f — Answer generation and the full query

The retrieved chunks are passed to the chat model as grounded context. Each chunk is labelled with its ID, section header and page, and the system prompt requires a citation for every claim and a fixed reply when the answer isn't in the chunks.

In [ ]:
def generate_answer(self, query: str, chunks: List[RetrievedChunk]) -> str:
    """
    Pass retrieved chunks as grounded context to the chat model.
    System prompt enforces citation and prohibits hallucination.
    Each chunk is labeled with its section header for traceability.
    """
    context = "\n\n".join([
        f"[{chunk.chunk_id} | {chunk.metadata.get('section_header', 'Unknown')} "
        f"| Page {chunk.metadata.get('page_start', '?')} "
        f"| Score {chunk.score:.4f}]\n{chunk.content}"
        for chunk in chunks
    ])

    system_prompt = """You are a compliance assistant for Walmart's US Product Quality
and Compliance Manual.

RULES:
1. Answer ONLY using the document chunks provided below.
2. Cite the chunk ID and section header for every claim e.g. [walmart-sec005 | F. Letters of Guarantee].
3. If the answer is not in the chunks, say exactly: "Not found in retrieved sections."
4. Use bullet points for lists. Be concise and precise.
5. Never invent information beyond what is explicitly in the chunks."""

    user_message = f"""RETRIEVED DOCUMENT CHUNKS:
{context}

QUESTION: {query}

Answer using only the chunks above. Cite chunk IDs and section headers."""

    response = self.openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0.1,
        max_tokens=700
    )

    return response.choices[0].message.content


def query(self, question: str, k: int = 4, verbose: bool = True) -> Dict[str, Any]:
    if verbose:
        print(f"\n🔍 Query: '{question}'")

    chunks  = self.hybrid_search(question, k=k)
    answer  = self.generate_answer(question, chunks)

    return {
        "question": question,
        "answer": answer,
        "sources": chunks
    }


WalmartRAGPipeline.generate_answer = generate_answer
WalmartRAGPipeline.query = query

## Step 6 — Result display

The "glass box" output: every source shows its chunk ID, section header, page, score and the method that found it, so every claim in the answer can be traced.

In [ ]:
def print_result(result: Dict[str, Any]) -> None:
    """
    Display a RAG result with full evidence tracing.
    Each source shows: chunk ID, section header, page number, score, method.
    This is the 'glass box' output — every claim is traceable.
    """
    print("\n" + "═" * 70)
    print(f"❓  {result['question']}")
    print("═" * 70)
    print(f"\n💬  ANSWER:\n{result['answer']}")

    print("\n📚  EVIDENCE TRAIL:")
    print(f"    {'Rank':<5} {'Chunk ID':<25} {'Section':<35} {'Page':<6} {'Score':<8} {'Method'}")
    print(f"    {'─'*4} {'─'*24} {'─'*34} {'─'*5} {'─'*7} {'─'*15}")

    for i, chunk in enumerate(result["sources"], 1):
        section = chunk.metadata.get("section_header", "Unknown")[:33]
        page    = str(chunk.metadata.get("page_start", "?"))
        print(
            f"    {i:<5} {chunk.chunk_id:<25} {section:<35} "
            f"{page:<6} {chunk.score:<8.4f} {chunk.retrieval_method}"
        )

    print("═" * 70)

## Build the pipeline

Connects to the collection and indexes the chunks. On the first run this embeds and uploads every chunk; on later runs (even in a new Colab session) the cloud collection is reused and only BM25 is rebuilt.

In [ ]:
pipeline = WalmartRAGPipeline(
    chroma_client,
    openai_api_key=os.environ["OPENAI_API_KEY"],
    reset=RESET_COLLECTION,
)
pipeline.index_documents(chunks)

print(f"\n✅ Pipeline ready — {len(chunks)} section-chunks indexed")

### Check the distance metric

`dense_search` turns distance into similarity with `1 − distance`, which is only correct in **cosine** space. This cell compares the distance Chroma Cloud returns with one computed by hand. If the collection was created earlier with a different metric (or the setting was ignored), the numbers won't match.

In [ ]:
import numpy as np

probe = "What happens if my product fails testing?"
res = pipeline.collection.query(
    query_texts=[probe], n_results=1, include=["embeddings", "distances"]
)

q = np.array(pipeline.embedding_fn([probe])[0], dtype=float)
d = np.array(res["embeddings"][0][0], dtype=float)
cos = float(q @ d / (np.linalg.norm(q) * np.linalg.norm(d)))
returned = float(res["distances"][0][0])

print(f"Distance returned by Chroma Cloud : {returned:.4f}")
print(f"1 − cosine similarity (by hand)   : {1 - cos:.4f}")
print(f"Squared L2 distance (by hand)     : {float(np.sum((q - d) ** 2)):.4f}")

if abs(returned - (1 - cos)) < 1e-3:
    print("\n✅ The collection uses cosine distance")
else:
    print("\n⚠️  Not cosine — set RESET_COLLECTION = True and rebuild the pipeline")

## Step 7 — Ask questions

### One question

In [ ]:
result = pipeline.query("What is the SKPPT program?")
print_result(result)

### Demo queries

Each query is labelled with the retriever expected to do best. Compare the **BM25 top** and **Dense top** lines printed above each answer.

In [ ]:
DEMO_QUERIES = [
    ("BM25 wins — exact acronym",    "What is the SKPPT program?"),
    ("BM25 wins — exact term",       "What is WERCS used for?"),
    ("Dense wins — natural language","What happens if my product fails testing?"),
    ("Dense wins — conceptual",      "How does Walmart handle product recalls?"),
    ("Hybrid wins — mixed",          "What are the Letter of Guarantee requirements?"),
]

for label, q in DEMO_QUERIES:
    print(f"\n── {label} ──")
    print_result(pipeline.query(q))

### Interactive chatbot

Type a question in the box that appears below the cell. Type `demo` to step through the demo queries one at a time, or `exit` to stop. **The cell keeps running until you type `exit`**, so other cells won't run until then.

In [ ]:
def run_chatbot(pipeline: WalmartRAGPipeline) -> None:
    print("\n" + "═" * 70)
    print("  WALMART COMPLIANCE RAG CHATBOT  (PDF-native, section-chunked)")
    print("  Chroma Cloud + BM25 + RRF + GPT-4o-mini")
    print("═" * 70)
    print("\n  Commands:")
    print("    ask any question about the Walmart compliance manual")
    print("    'demo'  → run 5 sample queries showing all retrieval strategies")
    print("    'exit'  → quit\n")

    while True:
        user_input = input("🧑 You: ").strip()

        if not user_input:
            continue

        if user_input.lower() in ("exit", "quit"):
            print("👋 Goodbye!")
            break

        if user_input.lower() == "demo":
            print("\n🚀 Running demo queries...\n")
            for label, q in DEMO_QUERIES:
                print(f"\n── {label} ──")
                result = pipeline.query(q)
                print_result(result)
                input("  Press Enter for next query...")
            continue

        result = pipeline.query(user_input)
        print_result(result)


run_chatbot(pipeline)

## Cleanup (optional)

The collection stays in Chroma Cloud after this notebook closes, and stored data counts toward your Chroma Cloud usage. Set `DELETE_CLOUD_COLLECTION = True` and run this cell to remove it.

In [ ]:
DELETE_CLOUD_COLLECTION = False

if DELETE_CLOUD_COLLECTION:
    chroma_client.delete_collection(COLLECTION_NAME)
    print(f"🗑️  Deleted '{COLLECTION_NAME}' from Chroma Cloud")
else:
    print(f"Kept '{COLLECTION_NAME}' in Chroma Cloud")

## Discussion: limitations worth raising

These are carried over unchanged from the script; they're good prompts for class discussion.

1. **Citations stop at the page range.** Step 1 extracts a bounding box for every span, but Step 2 drops it. Every sub-chunk of a long section also reports the *whole section's* page range, so a citation can't point to the exact page or line.
2. **Sub-chunks are fixed character windows.** Long sections are split every 1,200 characters, which can cut mid-sentence — the same problem the Step 2 notes criticize in character-based chunking.
3. **Header detection works span by span.** If PyMuPDF splits a header such as "III." and its title into separate spans, the `len(text) > 5` check rejects "III." and the section is missed. `NUMBERED_SUBSECTION_PATTERN` is defined but never used. Compare the detected section count with the manual's table of contents.
4. **BM25 always returns k results**, even when a chunk scores zero, so RRF can give an irrelevant chunk a fused score.
5. **The JSON cache is reused silently.** After changing the chunking code, delete `walmart_chunks.json` *and* set `RESET_COLLECTION = True`, or the old chunks stay in use.
6. **Cloud storage means third-party storage.** Every chunk's text now sits on Chroma's servers. That's fine for a public Walmart manual. It would not be fine for confidential material such as a deal data room.